In [3]:
import os
import numpy as np
import sympy as sp
import torch

from SymbolicMathematics.src.envs import set_cover_utils
from src.utils import AttrDict
from src.envs import build_env
from src.model import build_modules

from src.utils import to_cuda
from src.envs.sympy_utils import simplify


from src.envs import char_sp
from src.envs.char_sp import CharSPEnvironment
from itertools import combinations

import json
import matplotlib.pyplot as plt

## Required Functions

In [4]:
def is_covering(matrix, selected_columns, rows):
    covered = set()
    for col in selected_columns:
        covered.update(np.where(matrix[:, col])[0])
    return len(covered) == rows

In [5]:
def set_cover_bruteforce(matrix):
    rows, cols = matrix.shape
    for num_columns in range(1, cols + 1):
        for selected_columns in combinations(range(cols), num_columns):
            if is_covering(matrix, selected_columns, rows):
                return selected_columns
    return None

In [6]:
def list_to_tokens(lst):
    return [str(c) for c in str(lst) if str(c) != ' ']

In [7]:
def convert_token_sequence_to_list_of_lists(token_sequence):
    list_of_lists = []  # This will store the final list of lists
    current_list = []  # This will store the current list being constructed
    depth = 0  # Variable to keep track of the depth of nesting
    inner_list = []  # Initialize inner_list

    if len(token_sequence) == 0:
        return []

    for token in token_sequence:  # Iterate over each token in the token sequence
        if token == '[':  # If an opening bracket is encountered
            if depth > 0:
                inner_list = []  # Start a new inner list
            depth += 1  # Increment the depth of nesting
        elif token == ']':  # If a closing bracket is encountered
            depth -= 1  # Decrement the depth of nesting
            if depth == 0:
                if inner_list:  # Check if the inner list is not empty
                    current_list.append(inner_list)  # Append the inner list to the current list
                list_of_lists.append(current_list)  # Append the constructed list to the final result
                current_list = []  # Reset the current list
            else:
                current_list.append(inner_list)  # Append the inner list to the current list
                inner_list = []  # Start a new inner list
        elif token.isdigit():  # If the token is an integer
            inner_list.append(int(token))  # Convert token to integer and add to the inner list

    if len(list_of_lists) == 0:
        return []
    return list_of_lists[0]  # Return the final list of lists

In [118]:
seed = 125


def shuffle_lists(nested_list, rng):
    # Shuffle each inner list
    for inner_list in nested_list:
        rng.shuffle(inner_list)
    
    # Shuffle the outer list
    rng.shuffle(nested_list)

    return nested_list

# this is the weakness of the approx algorithm!
def gen_set_cover_long(max_universe_size=6, max_subsets=40, max_trues_per_subset=3):

    rng = np.random.RandomState(seed)
    universe_list = list(range(10))  # Default universe list from 0 to 10
    selected_elements = np.random.choice(universe_list, max_universe_size, replace=False)

    # Generate a random binary matrix
    num_rows = rng.randint(5, max_universe_size + 1) #elements
    num_cols = rng.randint(5, max_subsets + 1) #subsets
    binary_matrix = np.zeros((num_rows, num_cols), dtype=bool)

    for j in range(num_cols):
    # Choose random indices to set True, limited by max_trues_per_subset
        subset_size = rng.randint(1, max_trues_per_subset + 1)
        true_indices = rng.choice(num_rows, subset_size, replace=False)
        binary_matrix[true_indices, j] = True

    for i in range(num_rows):
        if not binary_matrix[i].any():
            binary_matrix[i, np.random.randint(0, num_cols)] = True
    for j in range(num_cols):
        if not binary_matrix[:, j].any():
            binary_matrix[np.random.randint(0, num_rows), j] = True

    # Remove duplicate rows
    binary_matrix = np.unique(binary_matrix, axis=1)
    # Find a set cover solution using brute force
    solution_indices = set_cover_bruteforce(binary_matrix)

    # Convert binary matrix columns to sequences of selected elements
    input_sequence = []
    for column in binary_matrix.T:
        subset = [selected_elements[row] for row in np.where(column)[0]]
        input_sequence.append(subset)
        
    print(input_sequence)
    # shuffle the list of lists
    input_sequence = shuffle_lists(input_sequence, rng)
    print(input_sequence)
    
    label_sequence = [subset for idx, subset in enumerate(input_sequence) if idx in solution_indices]

    input_sequence = list_to_tokens(input_sequence)
    label_sequence = list_to_tokens(label_sequence)


    return input_sequence, label_sequence

In [111]:
def validate_set_cover_solution(problem_set, solution_set, true_output):
    valid_solution_count = 0
    error_counts = {
        "Wrong Format": 0,
        "Subset not in problem set": 0,
        "Solution not covering problem set": 0,
        "Solution is not optimal": 0
    }
    coverage_percentage = 0

    # Check if the solution set is empty
    if not solution_set:
        error_counts["Wrong Format"] += 1
        return valid_solution_count, error_counts, coverage_percentage


    # Check if any subset in the solution set is empty
    for j, subset in enumerate(solution_set):
        if not subset:
            error_counts["Wrong Format"] += 1
            break

    # Check if all subsets in the solution set exist in the problem set
        if subset not in problem_set:
            error_counts["Subset not in problem set"] += 1
            break

    else:
        # Check if the solution set covers the entire problem set
        covered_tokens = set().union(*solution_set)
        problem_tokens = set(token for sublist in problem_set for token in sublist)
        if covered_tokens != problem_tokens:
            coverage_percentage = round(len(covered_tokens) / len(problem_tokens) * 100, 2)
            error_counts["Solution not covering problem set"] += 1

        else:
            if len(solution_set) > len(true_output):
                error_counts["Solution is not optimal"] += 1
            else:
                valid_solution_count += 1

    return valid_solution_count, error_counts, coverage_percentage

In [112]:
def set_cover_greedy(input_sequence):
    # Initialize variables
    universe = set()
    for subset in input_sequence:
        universe.update(subset)
    
    universe = list(universe)
    covered = set()
    selected_subsets = []

    # Iterate until all elements are covered
    while covered != set(universe):
        best_subset = None
        max_covered_elements = -1
        
        # Find the subset that covers the most uncovered elements
        for subset in input_sequence:
            uncovered_elements = set(subset) - covered
            if len(uncovered_elements) > max_covered_elements:
                max_covered_elements = len(uncovered_elements)
                best_subset = subset
        
        if best_subset is None:
            break
        
        # Add the best subset to the solution and update covered elements
        selected_subsets.append(best_subset)
        covered.update(best_subset)

    return selected_subsets


In [115]:
#Test
input1, label1 = gen_set_cover_long(6)
print("input sequence:", input1)
print("label sequence:", label1)

input sequence: ['[', '[', '7', ',', '2', ',', '8', ']', ',', '[', '9', ',', '7', ']', ',', '[', '0', ']', ',', '[', '9', ']', ',', '[', '8', ',', '4', ']', ',', '[', '9', ',', '0', ',', '7', ']', ',', '[', '8', ']', ',', '[', '8', ',', '9', ',', '2', ']', ',', '[', '7', ',', '8', ',', '0', ']', ',', '[', '9', ',', '8', ',', '4', ']', ',', '[', '2', ',', '4', ']', ',', '[', '8', ',', '9', ',', '7', ']', ',', '[', '2', ',', '7', ',', '9', ']', ',', '[', '4', ']', ',', '[', '0', ',', '9', ']', ',', '[', '2', ']', ',', '[', '8', ',', '2', ']', ',', '[', '9', ',', '8', ']', ',', '[', '7', ']', ',', '[', '2', ',', '0', ']', ',', '[', '4', ',', '2', ',', '9', ']', ',', '[', '4', ',', '2', ',', '8', ']', ']']
label sequence: ['[', '[', '2', ',', '7', ',', '9', ']', ',', '[', '7', ']', ']']


## Build environment / Reload model

In [59]:
# Reload Trained model
#model_path1 = './Model1/first_train/2spv1sq7hj/checkpoint.pth'
#assert os.path.isfile(model_path1)

model_path2 = './Model2/first_train/5r8wvsegfc/checkpoint.pth'
assert os.path.isfile(model_path2)


In [60]:
params = AttrDict({

    # environment parameters
    'env_name': 'char_sp',
    'int_base': 10,
    'balanced': False,
    'positive': True,
    'precision': 10,
    'n_variables': 1,
    'n_coefficients': 0,
    'leaf_probs': '0.75,0,0.25,0',
    'max_len': 512,
    'max_int': 5,
    'max_ops': 15,
    'max_ops_G': 15,
    'clean_prefix_expr': True,
    'rewrite_functions': '',
    #'tasks': 'prim_fwd',
    'tasks': 'set_cover',
    'operators': 'add:10,sub:3,mul:10,div:5,sqrt:4,pow2:4,pow3:2,pow4:1,pow5:1,ln:4,exp:4,sin:4,cos:4,tan:4,asin:1,acos:1,atan:1,sinh:1,cosh:1,tanh:1,asinh:1,acosh:1,atanh:1',

    # model parameters
    'cpu': True,
    'emb_dim': 128,
    'n_enc_layers': 2, #it was 6
    'n_dec_layers': 2, #it was 6
    'n_heads': 2, #it was 8
    'dropout': 0,
    'attention_dropout': 0,
    'sinusoidal_embeddings': False,
    'share_inout_emb': True,
    'reload_model': model_path2,

})

In [61]:
env = build_env(params)
#x = env.local_dict['x']

In [62]:
modules = build_modules(env, params)
encoder = modules['encoder']
decoder = modules['decoder']

In [63]:
problem_set_instance = [[1,2],[1,2,3],[2],[4]]
problem_instance_tokens = list_to_tokens(problem_set_instance)
print(problem_instance_tokens)

['[', '[', '1', ',', '2', ']', ',', '[', '1', ',', '2', ',', '3', ']', ',', '[', '2', ']', ',', '[', '4', ']', ']']


In [46]:
num_samples = 2
for i in range(num_samples):
    input_sequence, label_sequence = gen_set_cover2()
    input_set = convert_token_sequence_to_list_of_lists(input_sequence)
    label_set = convert_token_sequence_to_list_of_lists(label_sequence)
    approximation_set = set_cover_greedy(input_set)
    
    print("input:", input_set)
    print("label:", label_set)
    print("approximation:", approximation_set)


input: [[8], [2], [2, 8], [7, 2, 5, 8]]
label: [[7, 2, 5, 8]]
approximation: [[7, 2, 5, 8]]
input: [[2, 4], [7, 4, 5], [7, 2, 4], [7, 2, 4, 5]]
label: [[7, 2, 4, 5]]
approximation: [[7, 2, 4, 5]]


In [119]:
num_samples = 100
total_valid_answers = 0
total_not_optimal_answers = 0
valid_and_better_than_approximation = 0
valid_and_equal_to_approximation = 0
not_optimal_but_better_than_approximation = 0
not_optimal_and_equal_to_approximation = 0
not_optimal_and_worse_than_approximation = 0
approximation_answer_is_bigger_than_bruteforce = 0
approximation_answer_is_equal_to_bruteforce = 0


# Generate and test samples
for i in range(num_samples):
    input_sequence, label_sequence = gen_set_cover_long()

    # Encode the input sequence
    x1_prefix = input_sequence
    x1 = torch.LongTensor(
        [env.eos_index] +
        [env.word2id[w] for w in x1_prefix] +
        [env.eos_index]
    ).view(-1, 1)
    len1 = torch.LongTensor([len(x1)])
    # x1, len1 = to_cuda(x1, len1)  # Uncomment if you are using GPU
    with torch.no_grad():
        encoded = encoder('fwd', x=x1, lengths=len1, causal=False).transpose(0, 1)

    # Convert the input and label sequences to sets
    input_set = convert_token_sequence_to_list_of_lists(input_sequence)
    label_set = convert_token_sequence_to_list_of_lists(label_sequence)

    # Apply greedy algorithm to get the approximation sequence
    approximation_set = set_cover_greedy(input_set)

    # Decode with beam size 1
    with torch.no_grad():
        _, _, beam = decoder.generate_beam(encoded, len1, beam_size=1, length_penalty=1.0, early_stopping=1, max_len=200)
        assert len(beam) == 1
    hypotheses = beam[0].hyp
    assert len(hypotheses) == 1

    # Parse decoded hypothesis
    score, sent = hypotheses[0]
    ids = sent[1:].tolist()  # decoded token IDs
    tok = [env.id2word[wid] for wid in ids]  # convert to prefix

    try:
        model_output_set = convert_token_sequence_to_list_of_lists(tok)
    except:
        model_output_set = tok

    # Validate the model's output sequence
    valid_solution_count, error_counts, coverage_percentage = validate_set_cover_solution(input_set, model_output_set, label_set)
    
    print("input", input_set)
    print("True Answer, brute force", label_set)
    print("Model Output", model_output_set)
    print("Approximation", approximation_set)
    
    
    if len(approximation_set) > len(label_set):
        approximation_answer_is_bigger_than_bruteforce +=1
    if len(approximation_set) == len(label_set):
        approximation_answer_is_equal_to_bruteforce += 1
    

    #if valid_solution_count > 0 or error_counts["Solution is not optimal"] > 0:
    if valid_solution_count > 0:
        total_valid_answers += 1
        print("input", input_set)
        print("True Answer, brute force", label_set)
        print("Model Output", model_output_set)
        print("Approximation", approximation_set)
        if len(model_output_set) < len(approximation_set):
            valid_and_better_than_approximation += 1
            print("The solution is valid and bettfer than approximation")
        if len(model_output_set) == len(approximation_set):
            valid_and_equal_to_approximation +=1
            print("The solution is valid and equal to approximation")
            
            
        
    if error_counts["Solution is not optimal"] > 0:
        total_not_optimal_answers += 1
        print("input", input_set)
        print("True Answer, brute force", label_set)
        print("Model Output", model_output_set)
        print("Approximation", approximation_set)
        if len(model_output_set) < len(approximation_set):
            not_optimal_but_better_than_approximation += 1
            print("The solution is not optimal but better than approximation")
        if len(model_output_set) == len(approximation_set):
            not_optimal_and_equal_to_approximation +=1
            print("The solution is not optimal but similar to approximation")
        if len(model_output_set) > len(approximation_set):
            not_optimal_and_worse_than_approximation += 1
            print("The solution is not optimal but worse than approximation")
        

# Calculate percentages

print(f"Total valid answers: {total_valid_answers} out of {num_samples}")
print(f"among them {valid_and_better_than_approximation} are better than approximation")

print(f"Total not optimal answers: {total_not_optimal_answers} out of {num_samples}")
print(f"among them {not_optimal_but_better_than_approximation} are better than approximation and {not_optimal_and_worse_than_approximation} are worse than approximation")

print("approximation algorithm was not as good as bruteforce:", approximation_answer_is_bigger_than_bruteforce)
print("number of similar answers for approximation and brute force", approximation_answer_is_equal_to_bruteforce)


[[6], [5], [4], [4, 5], [0], [0, 6], [0, 5], [0, 4, 5], [7], [7, 6], [7, 5], [7, 4], [7, 4, 5], [7, 0, 5], [7, 0, 4], [2], [2, 5, 6], [2, 0], [2, 0, 6], [2, 0, 5], [2, 7, 5], [2, 7, 0]]
[[2, 7, 5], [0, 2], [6], [0], [5, 4], [0, 6, 2], [5], [5, 0, 7], [2, 5, 6], [0, 5, 4], [7, 4], [5, 0, 2], [7, 2, 0], [4], [6, 0], [7], [5, 7], [0, 5], [2], [7, 6], [4, 7, 0], [4, 7, 5]]
input [[2, 7, 5], [0, 2], [6], [0], [5, 4], [0, 6, 2], [5], [5, 0, 7], [2, 5, 6], [0, 5, 4], [7, 4], [5, 0, 2], [7, 2, 0], [4], [6, 0], [7], [5, 7], [0, 5], [2], [7, 6], [4, 7, 0], [4, 7, 5]]
True Answer, brute force [[7, 2, 0], [2]]
Model Output [[2, 7, 5], [6, 0, 4], [7, 0]]
Approximation [[2, 7, 5], [0, 6, 2], [5, 4]]
[[3], [5], [0], [0, 5], [1], [1, 3], [1, 5], [1, 0, 5], [9], [9, 3], [9, 5], [9, 0], [9, 0, 5], [9, 1, 5], [9, 1, 0], [8], [8, 5, 3], [8, 1], [8, 1, 3], [8, 1, 5], [8, 9, 5], [8, 9, 1]]
[[8, 9, 5], [1, 8], [3], [1], [5, 0], [1, 3, 8], [5], [5, 1, 9], [8, 5, 3], [1, 5, 0], [9, 0], [5, 1, 8], [9, 8, 1], [0

KeyboardInterrupt: Interrupted by user